# Nested Cross-Validation & Advanced Model Tuning

This notebook covers:

1. Why standard cross-validation is NOT enough
2. Nested cross-validation
3. Hyperparameter tuning in depth
4. Building reliable ML systems

Goal:

Understand how to evaluate models without bias.

## 1. The Core Problem

When we tune a model, we do:

    try parameters --> pick best

Problem:

We use the same data to:

- choose parameters  
- evaluate performance  

### Result

Performance becomes:

    optimistic (biased)


### Key Danger

We accidentally "overfit" to validation data

## 2. Why Standard Cross-Validation is Not Enough

### Standard workflow:

Cross-validation --> select best parameters  


### Problem

We evaluate:

    best model ON SAME DATA

### This means:

We are measuring:

    "How well did we optimize?"

Not:

    "How well does model generalize?"


## 3. Nested Cross-Validation

### Idea

Separate:

- model selection  
- model evaluation  


### Two loops:

OUTER loop:
    evaluates performance  

INNER loop:
    selects best parameters  

### Structure:

Outer CV:
    split data --> test model

Inner CV:
    optimize model on training set

### Key Insight

We simulate:

    "train model on training data → evaluate on unseen data"

## Nested Structure

Outer Loop:

    Train set --> INNER CV
    Test set  --> final evaluation

Inner Loop:

    try parameters → choose best

Flow:

data
--> split (outer)
--> optimize (inner)
--> evaluate (outer)

*now let's try this!*


In [ ]:
# NESTED CROSS-VALIDATION

import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

# data
X, y = load_iris(return_X_y=True)

# pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),   # normalize features
    ("pca", PCA()),                 # reduce dimensions
    ("clf", LogisticRegression())   # classifier
])

# hyperparameters
param_grid = {

    # PCA: control dimensionality
    "pca__n_components": [2, 3, 4],

    # regularization strength
    "clf__C": [0.1, 1, 10],

    # solver = optimization algorithm
    "clf__solver": ["lbfgs", "liblinear"]
}

# inner CV (model selection)
inner_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# outer CV (evaluation)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# grid search = INNER LOOP
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=inner_cv,
    scoring="accuracy"
)

# nested CV = OUTER LOOP
scores = cross_val_score(
    grid,
    X,
    y,
    cv=outer_cv
)

print("Nested CV scores:", scores)
print("Mean performance:", scores.mean())

## 4. What Happens Internally

For each outer split:

1. Split data --> train/test

2. INNER LOOP:
   - try parameter combinations
   - select best model

3. Train best model on full training set

4. Evaluate on TEST set (never seen before)

### Key Difference

Test data is NEVER used in tuning

### Result

Unbiased performance estimate

## 5. Hyperparameters: What and Why

### PCA

n_components:

- controls information compression
- tradeoff: information vs noise

### Logistic Regression

C:

- inverse of regularization strength
- small C --> strong regularization
- large C --> weak regularization

solver:

- optimization algorithm
- affects convergence and stability

### Important Insight

Hyperparameters control:

    model complexity + optimization behavior

In [ ]:
# EXTENDED HYPERPARAMETER SEARCH

param_grid = {

    "pca__n_components": [2, 3, 4],

    # stronger exploration
    "clf__C": [0.01, 0.1, 1, 10, 100],

    # try different solvers
    "clf__solver": ["lbfgs", "liblinear"],

    # max iterations → convergence
    "clf__max_iter": [100, 500, 1000]
}

## More Tuning Parameters

max_iter:

- controls number of optimization steps
- too small --> no convergence

learning process:

solver + iterations define

    how model finds optimal parameters

### Key Insight

Hyperparameters affect:

- speed
- stability
- final solution

## 6. Interpreting Nested CV Results

Mean score:

--> expected performance

Standard deviation:

--> stability

High variance:

--> model sensitive to data splits

### Important

Nested CV estimates:

    "true generalization performance"

## Common Mistakes

### 1. Using same data for tuning and testing

--> optimistic results

### 2. Scaling outside pipeline

--> data leakage

### 3. Too small dataset

--> unstable results

### 4. Over-tuning

--> model adapts to noise

### Key Insight

Evaluation must simulate real-world conditions

## Advanced Exercises

### Nested CV

1. Compare:
   - normal CV vs nested CV
   - which is higher? WHY?

### Hyperparameters

2. Add new parameters to grid:
   - penalty = "l1", "l2"
   - observe performance

### PCA

3. Use n_components=1 --> what happens?

### Stability

4. Print standard deviation:
   --> when is model unstable?

### Pipeline logic

5. Remove scaling --> what breaks?

### Critical Thinking

6. When is nested CV overkill?

### Bonus (Hard)

7. Replace LogisticRegression with:
   - RandomForest
   - compare tuning behavior